# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in fact_content_daily_performance = one content page's performance on one calendar day. My lane rolls these up to page-level over a mid-panel month, month=2026-03, rather than the _sample table (June 2026), since the _sample is the sealed final month and using it to build label logic would mean training on the same window I'd eventually need as an honest outcome check.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature	daily clicks, impressions, avg position, CTR, days since last content update — all knowable before the decision date
Label/proxy	a forward-looking outcome (e.g., traffic change in the following window) if one exists at this grain, or my w01/w02 staleness+demand+underperformance proxy computed from the same month
Context	content/page hash key, client hash key, month/date (used to group and filter, never fed to the model as a "feature" since it's an ID, not a signal)
Excluded	fact_content_query_90d (query-level hashed keywords) — dropped for this lane to avoid an unnecessary join and any risk of pulling in forward-looking query data; anything from dim_clients beyond gsc_data_start/ga4_data_start — excluded per the dataset's anonymization terms

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
import duckdb, os

con = duckdb.connect()
# In Colab: store your token as a Secret named HF_TOKEN, then:
from google.colab import userdata
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month, per the warning about the sealed final month

# 0) Inspect the real schema first — don't guess column names
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') LIMIT 1"))

# --- Query 1: grain check — one row really is one (content, day) ---
q1 = con.sql(f"""
    SELECT content_hash_id, report_date as date, COUNT(*) AS n
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print(f"Grain violations (should be 0 rows): {len(q1)}")

# --- Query 2: row count + date span for the slice ---
q2 = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print(q2)

# --- Query 3: availability, filtered with IS TRUE ---
# Swap 'is_available' for whichever boolean column the DESCRIBE above actually shows.
q3 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS surviving_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print(q3)
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_clicks)      AS avg_clicks_28d,       -- knowable: trailing daily clicks, no future data used
        AVG(gsc_impressions)  AS avg_impressions_28d, -- knowable: trailing daily impressions
        AVG(gsc_sum_position)     AS avg_position_28d,    -- knowable: trailing rank, observed daily
        AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) AS ctr_28d, -- knowable: derived from the two features above
        DATEDIFF('day', MAX(report_date), DATE '{MONTH}-01') AS days_since_update -- knowable: update timestamp precedes decision date
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY content_hash_id
""").df()
features_df.head(10)
# Deliberate leak: pull in something only knowable AFTER the decision (e.g. next month's clicks)
leaky_df = features_df.copy()
leaky_df["next_month_clicks"] = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_clicks) AS next_month_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_hash_id
""").df().set_index("content_hash_id").reindex(leaky_df["content_hash_id"]).values

# quick score with the leaked column in the feature set — watch it jump toward ~1.0
# ... fit a quick baseline model with and without next_month_clicks here ...

# then remove it and keep the honest, lower number
leaky_df = leaky_df.drop(columns=["next_month_clicks"])

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be 0 rows): 0
    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  surviving_rows
0     9841378         3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can never tell me why a page's performance changed — only that it did. The panel is unbalanced (dim_clients.gsc_data_start/ga4_data_start differ per client), so early rows for newer clients may be GSC-only with no GA4 signal, which could quietly bias any feature that assumes both sources exist. And because fact_content_query_90d uses salted, per-content aggregate shares for the rare tail (<10 impressions), any query-level rollup will undercount true long-tail volume rather than reporting zero, which is an easy thing to misread as "no demand."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.